In [ ]:
# Unsloth ve gerekli kütüphaneleri Colab ortamına kuruyoruz (Google wprkspace)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
!pip install --upgrade unsloth unsloth-zoo

In [ ]:
!pip install --upgrade --force-reinstall \
    "unsloth @ git+https://github.com/unslothai/unsloth.git" \
    "unsloth-zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True # 4-bit (Quantization)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("\nLlama 3 8B Colab GPU'suna başarıyla yüklendi!")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank ->> modelin öğrenme kapasitesi
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dream_catcher_dataset.jsonl", split="train")

alpaca_prompt = "### Instruction:\n{}\n### Input:\n{}\n### Response:\n{}"   # Alpaca Formatı

def format_func(ex):
    return {"text": [alpaca_prompt.format(i, inp, out) + tokenizer.eos_token for i, inp, out in zip(ex["instruction"], ex["input"], ex["output"])]}

dataset = dataset.map(format_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2, # Hız için 2 çekirdek kullan
    packing = False, # Optimizasyon için False bırak
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Sadece 60 adım eğiteceğiz
        learning_rate = 2e-4, # Öğrenme hızı "2e-4" ideal
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1, # Her adımda loss değerini gör
        optim = "adamw_8bit", # VRAM dostu AdamW optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
FastLanguageModel.for_inference(model) # Modeli eğitim modundan çıkarıp, 2 kat hızlı okuma moduna alır

test_prompt = alpaca_prompt.format(
    "Analyze this dream from a psychoanalytic perspective, utilizing Freudian and Jungian concepts.",
    "I was falling from a high building, but just before hitting the ground, I turned into a cat and landed safely.",
    "",
)

# Metni sayılara (tensor) çevirip ekran kartına yolluyoruz
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

# Model 256 kelimelik bir analiz üretiyor ve biz de bunu ekrana basıyoruz
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
!curl -I https://github.com

In [ ]:
!apt-get update && apt-get install -y cmake

In [ ]:
# Modeli ve adaptörleri birleştirip yerel kullanım için GGUF formatında paketliyoruz
model.save_pretrained_gguf("dream_catcher_model", tokenizer, quantization_method = "q4_k_m")

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp
!pip install gguf sentencepiece --break-system-packages --quiet
!cd /root/.unsloth/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && \
    cmake --build build --config Release --target llama-quantize -j$(nproc)
!python /root/.unsloth/llama.cpp/convert_hf_to_gguf.py \
    /content/dream_catcher_model \
    --outfile /content/dream_catcher_model_f16.gguf \
    --outtype f16

!/root/.unsloth/llama.cpp/build/bin/llama-quantize \
    /content/dream_catcher_model_f16.gguf \
    /content/dream_catcher_model_q4_k_m.gguf \
    q4_k_m

In [ ]:
!pip install torch==2.10.0 torchvision==0.25.0 --index-url https://download.pytorch.org/whl/cu128 --quiet

In [3]:
!pip install "transformers==5.5.0" --quiet

In [ ]:
import subprocess

result = subprocess.run([
    "python", "/root/.unsloth/llama.cpp/convert_hf_to_gguf.py",
    "/content/dream_catcher_model",
    "--outfile", "/content/dream_catcher_model_f16.gguf",
    "--outtype", "f16"
], capture_output=True, text=True)

print(result.stdout[-3000:])
print(result.stderr[-3000:])

In [ ]:
!/root/.unsloth/llama.cpp/build/bin/llama-quantize \
    /content/dream_catcher_model_f16.gguf \
    /content/dream_catcher_model_q4_k_m.gguf \
    Q4_K_M

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp /root/.unsloth/llama.cpp
!pip install -r /root/.unsloth/llama.cpp/requirements.txt

!python /root/.unsloth/llama.cpp/convert_hf_to_gguf.py \
    /content/dream_catcher_model \
    --outfile /content/dream_catcher_model.gguf \
    --outtype f16

# Quantize işlemi ile model 4-bit ler halinde yazılır, boyutunu azaltmaya yardımcı olur :)
!./root/.unsloth/llama.cpp/llama-quantize \
    /content/dream_catcher_model.gguf \
    /content/dream_catcher_model_q4_k_m.gguf \
    q4_k_m